In [21]:
import pandas as pd
from pathlib import Path
import os

In [12]:
df = pd.read_csv('ventas_enero.csv')
df

,fecha,producto,vendedor,region,cantidad,precio_unitario,total
0,2024-01-01,Hub USB,Matias Soto,Poniente,1,18000,18000
1,2024-01-02,Mouse,Matias Soto,Norte,4,22000,88000
2,2024-01-03,Auriculares,Matias Soto,Poniente,2,68000,136000
3,2024-01-04,Hub USB,Matias Soto,Poniente,5,18000,90000
4,2024-01-05,Teclado,Paula Mena,Norte,4,45000,180000
5,2024-01-06,Disco SSD,NaN,Norte,1,75000,75000
6,2024-01-07,Auriculares,Valeria Rios,Norte,1,68000,68000
7,2024-01-08,Auriculares,Matias Soto,Norte,3,68000,204000
8,2024-01-09,Hub USB,Diego Parra,Oriente,3,18000,54000
9,2024-01-10,Monitor,Matias Soto,Centro,2,290000,580000


In [7]:
print("Dimensiones del DataFrame: ", df.shape)
print("Tipos de datos: ", df.dtypes)
print("Valores faltantes por columna: ", df.isnull().sum())
display(df.describe())

df_filtrado = df[df['vendedor'].notna()]
print(f"Filas tras filtrar vendedores nulos: {len(df_filtrado)}")

Dimensiones del DataFrame:  (31, 7)
Tipos de datos:  fecha                str
producto             str
vendedor             str
region               str
cantidad           int64
precio_unitario    int64
total              int64
dtype: object
Valores faltantes por columna:  fecha              0
producto           0
vendedor           1
region             0
cantidad           0
precio_unitario    0
total              0
dtype: int64


,cantidad,precio_unitario,total
count,31.000000,31.000000,3.100000e+01
mean,2.838710,73483.870968,1.851290e+05
std,1.392762,75017.718337,2.134432e+05
min,1.000000,18000.000000,1.800000e+04
25%,2.000000,33500.000000,7.500000e+04
50%,3.000000,55000.000000,1.350000e+05
75%,4.000000,75000.000000,1.920000e+05
max,5.000000,290000.000000,1.160000e+06


Filas tras filtrar vendedores nulos: 30


In [14]:
df.isnull().sum()/len(df)*100

fecha              0.000000
producto           0.000000
vendedor           3.225806
region             0.000000
cantidad           0.000000
precio_unitario    0.000000
total              0.000000
dtype: float64

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   fecha            31 non-null     str  
 1   producto         31 non-null     str  
 2   vendedor         30 non-null     str  
 3   region           31 non-null     str  
 4   cantidad         31 non-null     int64
 5   precio_unitario  31 non-null     int64
 6   total            31 non-null     int64
dtypes: int64(3), str(4)
memory usage: 2.9 KB


In [ ]:
#Se puede observar que falta solo 1 dato (que corresponde al 3%) en cuanto a los demás tipos de datos se pueden dar en 
#la columna de arriba, con sus correspondientes tipos de datos.

In [ ]:
#
#
#

In [19]:
carpeta = Path('datos/ventas')
archivos = sorted(carpeta.glob('*.csv'))
print(f"Archivos CSV encontrados (glob): {len(archivos)}")

dfs = []
filas_individuales = 0

for archivo in archivos:
    df_temp = pd.read_csv(archivo, parse_dates=['fecha'])
    filas_individuales += len(df_temp)
    
    df_temp['mes'] = archivo.stem.replace('ventas_', '')
    dfs.append(df_temp)

consolidado = pd.concat(dfs, ignore_index=True)

print("Resultados de Validación")
print(f"Filas en el DataFrame consolidado: {len(consolidado)}")
print(f"Suma de filas individuales: {filas_individuales}")
print(f"Validación exitosa (Si / No): {'Si' if len(consolidado) == filas_individuales else 'No'}")
print(f"Columnas del DataFrame final: {list(consolidado.columns)}")

Archivos CSV encontrados (glob): 12
Resultados de Validación
Filas en el DataFrame consolidado: 397
Suma de filas individuales: 397
Validación exitosa (Si / No): Si
Columnas del DataFrame final: ['fecha', 'producto', 'vendedor', 'region', 'cantidad', 'precio_unitario', 'total', 'mes']


In [24]:
#Los archivos encontrados dentro de los csv son 12
#Las filas de dataframe consolidados son 397 según registro
#Suma de filas individuales como tal son 397 
#La validación fue éxitosa.
#Las columnas finales del dataframe datado son: "'fecha', 'producto', 'vendedor', 'region', 'cantidad', 
#'precio_unitario', 'total', 'mes'"

In [31]:
resumen_mensual = consolidado.groupby('mes')['total'].sum().reset_index()

resumen_productos = consolidado.groupby('producto').agg(
    cantidad_total=('cantidad', 'sum'),
    ingreso_total=('total', 'sum')
).reset_index()

prod_mas_vendido = resumen_productos.loc[resumen_productos['cantidad_total'].idxmax(), 'producto']
prod_mayor_ingreso = resumen_productos.loc[resumen_productos['ingreso_total'].idxmax(), 'producto']
promedio_transaccion = consolidado['total'].mean()


print("Métricas Calculadas")
print(f"Producto más vendido (cantidad): {prod_mas_vendido}")
print(f"Promedio de venta por transacción: ${promedio_transaccion:,.2f}")


Path('datos/salida').mkdir(parents=True, exist_ok=True)

consolidado.to_csv('datos/salida/ventas_2024_consolidado.csv', index=False, encoding='utf-8')

ruta_excel = 'datos/salida/reporte_anual_2024.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    consolidado.to_excel(writer, sheet_name='Datos_Completos', index=False)
    resumen_mensual.to_excel(writer, sheet_name='Resumen_Mensual', index=False)
    resumen_productos.to_excel(writer, sheet_name='Resumen_Productos', index=False)

with pd.ExcelFile(ruta_excel) as libro:
    print(f"\nHojas generadas en el Excel comprobadas: {libro.sheet_names}")

Métricas Calculadas
Producto más vendido (cantidad): Webcam
Promedio de venta por transacción: $543,584.38


ImportError: Pandas requires version '3.1.5' or newer of 'openpyxl' (version '3.1.2' currently installed).

In [51]:
#Total de ventas CLP por mes: 
total=mean_ventas=consolidado['total'].sum()
print('total ventas por mes', total)
#El producto más vendido es: Webcam
producto_mas_vendido = nombre_producto = resumen_productos.loc[resumen_productos['cantidad_total'].idxmax(), 'producto']
print('producto más vendido: ', producto_mas_vendido)
#Mes con mayor ingreso es: ''Diciembre'
mes_mayor=consolidado.groupby('mes')['total'].sum().idxmax()
print('Mes con mayor ingreso: ', mes_mayor)
#El promedio de ventas por transaccion fue de $543,584.38 CLP
mean_ventas=consolidado['total'].mean()
print('Promedio de ventas: ', mean_ventas)

total ventas por mes 215803000
producto más vendido:  Webcam
Mes con mayor ingreso:  diciembre
Promedio de ventas:  543584.3828715365


In [42]:
ruta = Path('datos/ventas/ventas_enero.CSV')
if ruta.exists():
    df_caso1 = pd.read_csv(ruta)
    print('Caso 1 Archivo encontrado/En caso que no arroja un else de no encontrado')
else:
    print('Caso 1 Archivo no encontrado:', ruta.resolve())

carpeta_erronea = Path('datos/ventas_2025')
archivos_erroneos = list(carpeta_erronea.glob('*.csv'))
if len(archivos_erroneos) == 0:
    print('Caso 2 No se encontraron archivos en:', carpeta_erronea)
else:
    resultado = pd.concat([pd.read_csv(a) for a in archivos_erroneos], ignore_index=True)

df_caso3 = pd.read_csv('datos/archivo_cp1252.csv', encoding='cp1252')
print("Caso 3 Archivo cp1252 leído correctamente con encoding='cp1252'.")

dfs_corregido = []
for archivo in sorted(Path('datos/ventas').glob('*.csv')):
    df_loop = pd.read_csv(archivo)
    df_loop['mes'] = archivo.stem 
    dfs_corregido.append(df_loop)
print(f"Caso 4 - Consolidado corregido. Se procesaron {len(dfs_corregido)} archivos correctamente.")

Caso 1 Archivo encontrado/En caso que no arroja un else de no encontrado
Caso 2 No se encontraron archivos en: datos\ventas_2025
Caso 3 Archivo cp1252 leído correctamente con encoding='cp1252'.
Caso 4 - Consolidado corregido. Se procesaron 12 archivos correctamente.


CASO 1
FileNotFoundError: [Errno 2] No such file 
or directory: 'datos/ventas/ventas_enero.CSV'

El error ocurre porque los sistemas de archivos son sensibles a las mayusculas y minusculas

CASO 2
ValueError: No objects to concatenate

Al apuntar a una carpeta inexistente o vacía (datos/ventas_2025), el método .glob() no encuentra nada y la comprensión de listas genera una lista vacía []

CASO 3
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 15: invalid continuation byte

Por defecto, pd.read_csv() asume que los archivos están codificados en utf-8.4
Lo cual significa que posee a lo mejor ñ o tildes las cuales pandas tiene problemas
al leer al no ser declaradas como tal con un encoding='cp1252' para no tener problemas

CASO 4
No genera error por depuración/consola, si no es más que nada
un error lógico que no muestra.

Al dejar la línea 
df['mes'] = archivo.stem sin indentar, esta queda fuera del ciclo for.